<a href="https://colab.research.google.com/github/cristiangaymartin/tfm-estados-mercado/blob/main/notebooks/06_snowflake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install snowflake-connector-python --quiet
%pip install "snowflake-connector-python[pandas]" --quiet

In [2]:
from google.colab import userdata
from snowflake.connector.pandas_tools import write_pandas
import snowflake.connector, os
import pandas as pd

In [3]:
conn = snowflake.connector.connect(
    user=userdata.get("SNOWFLAKE_USER"),
    password=userdata.get("SNOWFLAKE_PASSWORD"),
    account=userdata.get("SNOWFLAKE_ACCOUNT"),
    warehouse="SNOWFLAKE_LEARNING_WH",
    database="TFM_MERCADOS",
    schema="DATOS"
)
cur = conn.cursor()
cur.execute("SELECT CURRENT_VERSION()")
print("Conexión exitosa. Versión:", cur.fetchone()[0])
cur.close()

Conexión exitosa. Versión: 10.30.102


True

In [4]:
RUTA_CONGELADO = "/content/drive/MyDrive/TFM/data_congelado"

archivos = [f for f in os.listdir(RUTA_CONGELADO) if f.startswith("mercado_") and f.endswith(".csv")]
print(f"Mercados a subir: {len(archivos)}\n")

for archivo in sorted(archivos):
    # Nombre de la tabla: mercado_SP500.csv -> SP500
    nombre_tabla = archivo.replace("mercado_", "").replace(".csv", "").upper()

    # Cargar el CSV congelado
    df = pd.read_csv(f"{RUTA_CONGELADO}/{archivo}", parse_dates=[0])
    df = df.rename(columns={df.columns[0]: "FECHA"})   # la primera columna es la fecha

    # Crear la tabla y subir los datos
    exito, n_chunks, n_filas, _ = write_pandas(
        conn, df, nombre_tabla,
        auto_create_table=True, overwrite=True
    )
    print(f"{nombre_tabla:12s}: {'OK' if exito else 'ERROR'} — {n_filas} filas subidas")

print("\nCarga completada.")

Mercados a subir: 12

BONOUSA     : OK — 5645 filas subidas
CHINA       : OK — 6663 filas subidas
FTSE100     : OK — 8840 filas subidas
IBEX35      : OK — 7973 filas subidas
JAKARTA     : OK — 8468 filas subidas
KOSPI       : OK — 6911 filas subidas
NIFTY50     : OK — 4238 filas subidas
NIKKEI225   : OK — 8592 filas subidas
ORO         : OK — 6106 filas subidas
SP500       : OK — 8817 filas subidas
STOXX600    : OK — 5204 filas subidas
TAIEX       : OK — 6745 filas subidas

Carga completada.


In [5]:
# Una consulta SQL: traer los datos del S&P desde Snowflake
consulta = """
    SELECT *
    FROM SP500
    ORDER BY FECHA
"""

# Ejecutamos la consulta y traemos el resultado a un DataFrame de pandas
cur = conn.cursor()
cur.execute(consulta)
df_sp = cur.fetch_pandas_all()
cur.close()

print("Datos traídos desde Snowflake:", df_sp.shape)
print("Rango de fechas:", df_sp["FECHA"].min(), "→", df_sp["FECHA"].max())
df_sp.head()

Datos traídos desde Snowflake: (8817, 4)
Rango de fechas: 631238400000000000 → 1735603200000000000


,FECHA,Close,ret_log,vol_21d
0,631238400000000000,359.690002,NaN,NaN
1,631324800000000000,358.760010,-0.002589,NaN
2,631411200000000000,355.670013,-0.008650,NaN
3,631497600000000000,352.200012,-0.009804,NaN
4,631756800000000000,353.790009,0.004504,NaN


In [10]:
cur = conn.cursor()
cur.execute('SELECT * FROM SP500 LIMIT 3')
filas = cur.fetchall()
nombres = [d[0] for d in cur.description]
cur.close()
print("Columnas:", nombres)
for f in filas:
    print(f)

Columnas: ['FECHA', 'Close', 'ret_log', 'vol_21d']
(631238400000000000, 359.69000244140625, None, None)
(631324800000000000, 358.760009765625, -0.0025888876871238, None)
(631411200000000000, 355.6700134277344, -0.0086502960559933, None)


In [12]:
consulta_por_anio = """
    SELECT
        YEAR(TO_TIMESTAMP(FECHA / 1000000000)) AS anio,
        COUNT(*) AS dias,
        ROUND(AVG("Close"), 2) AS precio_medio,
        ROUND(MIN("Close"), 2) AS minimo,
        ROUND(MAX("Close"), 2) AS maximo
    FROM SP500
    GROUP BY YEAR(TO_TIMESTAMP(FECHA / 1000000000))
    ORDER BY anio
"""
cur = conn.cursor()
cur.execute(consulta_por_anio)
df_anios = cur.fetch_pandas_all()
cur.close()
print("Estadísticas anuales del S&P (fecha convertida en SQL):")
df_anios.head(10)

Estadísticas anuales del S&P (fecha convertida en SQL):


,ANIO,DIAS,PRECIO_MEDIO,MINIMO,MAXIMO
0,1990,253,334.63,295.46,368.95
1,1991,253,376.19,311.49,417.09
2,1992,254,415.75,394.50,441.28
3,1993,253,451.61,429.05,470.94
4,1994,252,460.42,438.92,482.00
5,1995,252,541.72,459.11,621.69
6,1996,254,670.49,598.48,757.03
7,1997,253,873.43,737.01,983.79
8,1998,252,1085.50,927.69,1241.81
9,1999,252,1327.33,1212.19,1469.25
